# BookMind: дообучение Qwen3-4B на туркменских примерах

Блокнот для **Kaggle** (видеокарта T4 ×1 или ×2, интернет включён).

Перед запуском:
1. Справа **Settings → Accelerator → GPU T4**, **Internet → On**.
2. **Add Input → Upload** — загрузите `train.jsonl` и `val.jsonl` из `D:\llm\train\out\` как один датасет с именем `bookmind-tm-train`, видимость **Private**.
3. Вверху **Save Version → Save & Run All (Commit)**. Так блокнот работает на сервере, даже если закрыть браузер. Обучение занимает около 2–4 часов.
4. Когда версия закончится, откройте её → вкладка **Output** → скачайте `bookmind-tm-Q4_K_M.gguf` (~2.5 ГБ) в `D:\llm\models\` и папку `bookmind-tm-lora`.


In [ ]:
%%capture
!pip install -q unsloth


In [ ]:
import glob, json, os
# Где Kaggle положил загруженный датасет (имя папки может немного отличаться).
DATA = os.path.dirname(glob.glob('/kaggle/input/**/train.jsonl', recursive=True)[0])
print('данные:', DATA)


In [ ]:
from unsloth import FastLanguageModel

# Та же модель, что в программе (Qwen/Qwen3-4B). Примеры длинные: туркменский текст
# даёт ~5700 токенов на пример, поэтому длина 7168.
MAX_LEN = 7168
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-4B',
    max_seq_length = MAX_LEN,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)


In [ ]:
from datasets import load_dataset

# Программа отключает «размышления» Qwen3 (enable_thinking=False) — учим в том же виде.
def to_text(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, enable_thinking=False)}

train = load_dataset('json', data_files=f'{DATA}/train.jsonl', split='train').map(to_text)
val = load_dataset('json', data_files=f'{DATA}/val.jsonl', split='train').map(to_text)
print(len(train), 'примеров для обучения,', len(val), 'для проверки')
print(train[0]['text'][-600:])


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train,
    eval_dataset = val,
    args = SFTConfig(
        dataset_text_field = 'text',
        max_length = MAX_LEN,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 2,
        learning_rate = 1e-4,          # поменьше, чтобы не испортить русский и английский
        warmup_ratio = 0.05,
        lr_scheduler_type = 'cosine',
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        logging_steps = 10,
        eval_strategy = 'steps',
        eval_steps = 50,
        save_strategy = 'no',
        seed = 3407,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)
# Учим только ответу модели, а не тексту книги и вопросу.
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part = '<|im_start|>assistant\n',
)


In [ ]:
stats = trainer.train()
print(stats)


In [ ]:
# Быстрая проверка на трёх примерах, которых модель не видела.
FastLanguageModel.for_inference(model)
for ex in val.select(range(3)):
    msgs = ex['messages'][:-1]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=200, temperature=0.2, do_sample=True)
    print('ВОПРОС:', msgs[-1]['content'].split('Sorag:')[-1].strip())
    print('ОЖИДАЛОСЬ:', ex['messages'][-1]['content'])
    print('МОДЕЛЬ:', tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip())
    print('---')


In [ ]:
# Сохраняем адаптер (маленький, на всякий случай) и готовую модель GGUF Q4_K_M для llama.cpp.
model.save_pretrained('bookmind-tm-lora')
tokenizer.save_pretrained('bookmind-tm-lora')
model.save_pretrained_gguf('bookmind-tm', tokenizer, quantization_method='q4_k_m')


In [ ]:
import shutil
# Unsloth кладёт файл в папку bookmind-tm (или рядом) — переносим под понятным именем.
found = [p for p in glob.glob('**/*.gguf', recursive=True) if 'Q4_K_M' in p.upper()]
print(found)
shutil.move(found[0], '/kaggle/working/bookmind-tm-Q4_K_M.gguf')
print(os.path.getsize('/kaggle/working/bookmind-tm-Q4_K_M.gguf') / 1e9, 'ГБ')
